# Classification — EfficientNet-B3 Multi-Head 속성 분류기

- 목적: YOLO crop → 색상(`color_class1`) + 형태(`drug_shape`) 속성 분류 (2-head)
- 모델: EfficientNet-B3 (timm, pretrained, 300px)
- Bbox 소스: `yolo_pred` 기준 (GT crop 제외)
- 체크포인트 기준: mean(val macro-F1_color, val macro-F1_shape) 최대값
- **Split:** train crop manifest 80/20 내부 분할 (StratifiedGroupKFold, group=image_file)

## 0. Mount / Imports / Config

In [1]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()

Mounted at /content/drive


In [2]:
!pip install timm scikit-learn -q

In [3]:
import gc
import json
import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from IPython.display import display
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 50)

In [4]:
# ── 경로 상수 ──────────────────────────────────────────────────────────────────
DRIVE_BASE    = Path('/content/drive/MyDrive/Conference/Pillot')
CACHE_DIR     = DRIVE_BASE / 'cache'
CROP_DIR      = CACHE_DIR / 'crops'
CROP_PROG_DIR = CACHE_DIR / 'crop_progress'
YOLO_BBOX_DIR = CACHE_DIR / 'yolo_bbox'
CKPT_DIR      = CACHE_DIR / 'classification' / 'jh_effb3_300'
ENCODER_DIR   = CKPT_DIR

TRAIN_MANIFEST_PATH = CROP_PROG_DIR / 'train_crop_manifest.csv'
TRAIN_YOLO_CSV      = YOLO_BBOX_DIR / 'train_yolo_pred.csv'

LOCAL_CROP_ROOT = Path('/content/crops_local')

# ── 하이퍼파라미터 ──────────────────────────────────────────────────────────────
INPUT_SIZE   = 300
BATCH_SIZE   = 32
NUM_EPOCHS   = 15
LR           = 1e-4
WEIGHT_DECAY = 1e-4
SEED         = 1213
VAL_RATIO    = 0.2
SPLIT_SEED   = 42

# ── 재현성 ────────────────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

CKPT_DIR.mkdir(parents=True, exist_ok=True)

device: cuda


## 0.5 이미지 로컬 복사 (Drive I/O 병목 방지)

Drive에서 직접 PNG를 읽으면 학습 중 ~19만 회 Drive 접근 → 속도 5~20× 저하.  

`LOCALIZE=True`로 한 번 복사 후, 이후 재시작 시에는 `LOCALIZE=False`로 스킵 가능.

In [5]:
LOCALIZE = True

if LOCALIZE:
    print('Copying crops from Drive to local SSD...')
    shutil.copytree(str(CROP_DIR), str(LOCAL_CROP_ROOT), dirs_exist_ok=True)
    print(f'Done. Local root: {LOCAL_CROP_ROOT}')
else:
    if not LOCAL_CROP_ROOT.exists():
        raise RuntimeError('LOCAL_CROP_ROOT does not exist. Set LOCALIZE=True and re-run.')
    print(f'Skipping copy. Using existing: {LOCAL_CROP_ROOT}')


def remap_crop_path(p: str) -> str:
    """drive/MyDrive/.../cache/crops/{split}/... → crops_local/{split}/..."""
    suffix = p.split('cache/crops/')[-1]
    return str(LOCAL_CROP_ROOT / suffix)

Copying crops from Drive to local SSD...
Done. Local root: /content/crops_local


## 1. Manifest 로드 및 Train/Val Split

- 단일 train crop manifest 로드 → 필터링 → StratifiedGroupKFold 80/20 분할
- group key: `image_file` (같은 이미지에서 나온 crop이 train/val에 섞이지 않도록)
- 분할 전 필터 순서: `yolo_pred` → `has_detection` → `label_map`

In [6]:
from sklearn.model_selection import StratifiedGroupKFold

manifest_df = pd.read_csv(TRAIN_MANIFEST_PATH)
print(f'raw rows: {len(manifest_df):,}')

if TRAIN_YOLO_CSV.exists():
    yolo_df = pd.read_csv(TRAIN_YOLO_CSV)[['image_file', 'has_detection']]
    manifest_df = manifest_df.merge(yolo_df, on='image_file', how='left')
    manifest_df['has_detection'] = manifest_df['has_detection'].fillna(True)
    no_det = int((manifest_df['has_detection'] == False).sum())
    manifest_df = manifest_df[manifest_df['has_detection'] == True].copy()
    print(f'  Dropped has_detection=False: {no_det:,}')

manifest_df = manifest_df[manifest_df['source_bbox'] == 'yolo_pred'].copy()
manifest_df['crop_path'] = manifest_df['crop_path'].map(remap_crop_path)
print(f'  After yolo_pred filter: {len(manifest_df):,} rows')

raw rows: 12,852
  Dropped non-yolo_pred: 0
  Dropped has_detection=False: 0


NameError: name 'label_map' is not defined

## 3. Attribute Encoder + DB JOIN + Train/Val Split

- MySQL `drug_master`에서 `color_class1`(30 classes), `drug_shape`(11 classes) 조회
- manifest CSV 파일은 변경 없음 — 메모리 DataFrame만 보강
- Split stratify 기준: `color_class1 + '_' + drug_shape` 조합 문자열

In [ ]:
import pymysql

conn = pymysql.connect(
    host='103.218.161.72', port=3306,
    user='zuhyeong_admin', password='1234', database='pilliot_db'
)
attr_df = pd.read_sql(
    "SELECT item_seq, color_class1, drug_shape FROM drug_master "
    "WHERE color_class1 IS NOT NULL AND color_class1 != '' "
    "AND drug_shape IS NOT NULL AND drug_shape != ''",
    conn
)
conn.close()

df = manifest_df.merge(attr_df, on='item_seq', how='inner')
print(f'manifest: {len(manifest_df):,}행  →  속성 JOIN 후: {len(df):,}행  '
      f'(제거: {len(manifest_df)-len(df):,}행)')

color_classes = sorted(df['color_class1'].unique())
shape_classes = sorted(df['drug_shape'].unique())
color_to_idx  = {c: i for i, c in enumerate(color_classes)}
shape_to_idx  = {s: i for i, s in enumerate(shape_classes)}

NUM_COLORS = len(color_classes)
NUM_SHAPES = len(shape_classes)
print(f'color: {NUM_COLORS} classes  shape: {NUM_SHAPES} classes')

ENCODER_DIR.mkdir(parents=True, exist_ok=True)
(ENCODER_DIR / 'color_encoder.json').write_text(
    json.dumps(color_to_idx, ensure_ascii=False, indent=2))
(ENCODER_DIR / 'shape_encoder.json').write_text(
    json.dumps(shape_to_idx, ensure_ascii=False, indent=2))

df['color_idx']      = df['color_class1'].map(color_to_idx)
df['shape_idx']      = df['drug_shape'].map(shape_to_idx)
df['color_shape_key'] = df['color_class1'] + '_' + df['drug_shape']

sgkf = StratifiedGroupKFold(n_splits=round(1 / VAL_RATIO), shuffle=True, random_state=SPLIT_SEED)
train_idx, val_idx = next(sgkf.split(df, df['color_shape_key'], df['image_file']))

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[val_idx].reset_index(drop=True)

leaked = set(train_df['image_file']) & set(val_df['image_file'])
assert len(leaked) == 0, f'image_file leakage: {len(leaked)} files'

print(f'train: {len(train_df):,} rows  val: {len(val_df):,} rows')

## 5. Dataset 클래스

In [ ]:
class PillCropDataset(Dataset):
    def __init__(self, df, transform):
        self.records = df[['crop_path', 'color_idx', 'shape_idx']].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        img = Image.open(row['crop_path']).convert('RGB')
        return self.transform(img), int(row['color_idx']), int(row['shape_idx'])

## 6. Augmentation

속성 분류기(색상·형태)는 orientation-invariant → flip 허용.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(INPUT_SIZE, scale=(0.75, 1.0),
                                 interpolation=InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomAffine(degrees=(-15, 15), translate=(0.05, 0.05)),
    transforms.RandomPerspective(distortion_scale=0.15, p=0.3),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),
])

val_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE), interpolation=InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = PillCropDataset(train_df, train_transform)
val_ds   = PillCropDataset(val_df,   val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'train: {len(train_ds):,} samples  val: {len(val_ds):,} samples')

## 7. 모델

EfficientNet-B3 backbone + 2-head (color 30 classes, shape 11 classes)

In [ ]:
class PillAttributeNet(nn.Module):
    def __init__(self, num_colors, num_shapes):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b3', pretrained=True, num_classes=0)
        feat = self.backbone.num_features
        self.head_color = nn.Linear(feat, num_colors)
        self.head_shape = nn.Linear(feat, num_shapes)

    def forward(self, x):
        f = self.backbone(x)
        return self.head_color(f), self.head_shape(f)

model = PillAttributeNet(NUM_COLORS, NUM_SHAPES).to(device)
print(f'backbone features: {model.backbone.num_features}')
print(f'head_color: {NUM_COLORS} classes  head_shape: {NUM_SHAPES} classes')

## 8. 학습 루프

- Optimizer: AdamW, Cosine Annealing LR
- Loss: CE_color + CE_shape
- AMP: T4 기준 ~2× 속도, OOM 방지
- Checkpoint: `best.pt` (mean macro-F1 기준) + `latest.pt` (매 epoch, resume용)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler    = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))

best_score = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    total_loss = 0.0

    for imgs, y_color, y_shape in tqdm(train_loader, desc=f'E{epoch}/{NUM_EPOCHS} train', leave=False):
        imgs    = imgs.to(device)
        y_color = y_color.to(device)
        y_shape = y_shape.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
            out_c, out_s = model(imgs)
            loss = criterion(out_c, y_color) + criterion(out_s, y_shape)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    # ── Val ──────────────────────────────────────────────────────────────────
    model.eval()
    all_c_true, all_c_pred = [], []
    all_s_true, all_s_pred = [], []

    with torch.no_grad():
        for imgs, y_color, y_shape in tqdm(val_loader, desc=f'E{epoch}/{NUM_EPOCHS} val', leave=False):
            out_c, out_s = model(imgs.to(device))
            all_c_true.extend(y_color.numpy())
            all_c_pred.extend(out_c.argmax(1).cpu().numpy())
            all_s_true.extend(y_shape.numpy())
            all_s_pred.extend(out_s.argmax(1).cpu().numpy())

    f1_color = f1_score(all_c_true, all_c_pred, average='macro', zero_division=0)
    f1_shape = f1_score(all_s_true, all_s_pred, average='macro', zero_division=0)
    score    = (f1_color + f1_shape) / 2

    scheduler.step()
    print(f'E{epoch:02d} | loss={total_loss/len(train_loader):.4f} | '
          f'f1_color={f1_color:.4f} | f1_shape={f1_shape:.4f} | mean={score:.4f}')

    torch.save(model.state_dict(), CKPT_DIR / 'latest.pt')
    if score > best_score:
        best_score = score
        torch.save(model.state_dict(), CKPT_DIR / 'best.pt')
        meta = {'epoch': epoch, 'f1_color': round(f1_color, 6),
                'f1_shape': round(f1_shape, 6), 'mean_f1': round(score, 6)}
        (CKPT_DIR / 'best_meta.json').write_text(json.dumps(meta, indent=2))
        print(f'  → best saved (mean_f1={score:.4f})')

print(f'\nTraining done. Best mean macro-F1: {best_score:.4f}')

## 9. 평가 (best.pt 기준)

head별 macro-F1 / classification_report

In [ ]:
model.load_state_dict(torch.load(CKPT_DIR / 'best.pt', map_location=device))
model.eval()

all_c_true, all_c_pred = [], []
all_s_true, all_s_pred = [], []

with torch.no_grad():
    for imgs, y_color, y_shape in tqdm(val_loader, desc='Eval'):
        out_c, out_s = model(imgs.to(device))
        all_c_true.extend(y_color.numpy())
        all_c_pred.extend(out_c.argmax(1).cpu().numpy())
        all_s_true.extend(y_shape.numpy())
        all_s_pred.extend(out_s.argmax(1).cpu().numpy())

f1_color = f1_score(all_c_true, all_c_pred, average='macro', zero_division=0)
f1_shape = f1_score(all_s_true, all_s_pred, average='macro', zero_division=0)
print(f'Val macro-F1  color={f1_color:.4f}  shape={f1_shape:.4f}  mean={((f1_color+f1_shape)/2):.4f}')

In [ ]:
print('=== Color Head ===')
print(classification_report(all_c_true, all_c_pred,
      target_names=color_classes, zero_division=0))

print('=== Shape Head ===')
print(classification_report(all_s_true, all_s_pred,
      target_names=shape_classes, zero_division=0))

In [ ]:
from sklearn.metrics import confusion_matrix

for title, y_true, y_pred, classes in [
    ('Color', all_c_true, all_c_pred, color_classes),
    ('Shape', all_s_true, all_s_pred, shape_classes),
]:
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    fig, ax = plt.subplots(figsize=(max(8, len(classes)), max(8, len(classes))))
    im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(classes)))
    ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(classes, fontsize=8)
    ax.set_title(f'{title} Confusion Matrix')
    plt.colorbar(im, ax=ax, fraction=0.03)
    plt.tight_layout()
    plt.show()

## 10. Failure Analysis — pad_fraction

`pad_fraction`이 높은 샘플이 오분류에 몰리면 classifier 문제가 아닌 **crop framing 문제**일 수 있습니다.

In [ ]:
result_df = pd.DataFrame({
    'correct_color': [p == l for p, l in zip(all_c_pred, all_c_true)],
    'correct_shape': [p == l for p, l in zip(all_s_pred, all_s_true)],
    'pad_fraction':  val_df['pad_fraction'].values,
})
result_df['correct_both'] = result_df['correct_color'] & result_df['correct_shape']

print('Val pad_fraction distribution:')
display(result_df['pad_fraction'].describe())

if result_df['pad_fraction'].max() < 1e-6:
    print('\npad_fraction이 모두 0 → 이 zip에서는 pad_fraction 분석 불가.')
else:
    bins = [0.0, 0.05, 0.1, 0.2, 0.5, 1.01]
    result_df['pad_bin'] = pd.cut(result_df['pad_fraction'], bins=bins, right=False)
    pad_analysis = result_df.groupby('pad_bin')[['correct_color', 'correct_shape', 'correct_both']].mean()
    print('\n--- Accuracy by pad_fraction bin ---')
    display(pad_analysis)

## 11. (Optional) 224 vs 300 비교

`COMPARE_224 = True`로 바꾸면 동일 조건으로 224px 모델을 추가 학습합니다.

In [ ]:
COMPARE_224 = False

if COMPARE_224:
    INPUT_SIZE_224 = 224
    CKPT_DIR_224   = CACHE_DIR / 'classification' / 'jh_effb3_224'
    CKPT_DIR_224.mkdir(parents=True, exist_ok=True)

    train_transform_224 = transforms.Compose([
        transforms.Resize((INPUT_SIZE_224, INPUT_SIZE_224), interpolation=InterpolationMode.BILINEAR),
        transforms.RandomAffine(degrees=(-15, 15), translate=(0.05, 0.05), scale=(0.95, 1.05)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    val_transform_224 = transforms.Compose([
        transforms.Resize((INPUT_SIZE_224, INPUT_SIZE_224), interpolation=InterpolationMode.BILINEAR),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    # Section 8 학습 루프 동일하게 재사용 (CKPT_DIR → CKPT_DIR_224 교체)
    print('COMPARE_224=True: Section 8 루프를 복사하고 CKPT_DIR를 CKPT_DIR_224로 교체하세요.')
else:
    print('COMPARE_224=False: skipped.')